<a href="https://colab.research.google.com/github/nami-04/E-Commerce-Sales-Customer-Analytics/blob/main/Ecommerce_Analytics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# E-Commerce Analytics

- 2,000 customers
- 60 products
- 20,000 orders
- revenue, cost, profit and discounts
- customer segments
- product/category/brand analysis
- payment method and sales channel
- returns and cancellations
- BigQuery views ready for Data Studio

**Project:** `ecommerce-analytics-505808`  
**Dataset:** `ecommerce_analytics`



In [ ]:
!pip -q install google-cloud-bigquery pandas numpy db-dtypes

from google.colab import auth
auth.authenticate_user()

from google.cloud import bigquery
import pandas as pd
import numpy as np

PROJECT_ID = "ecommerce-analytics-505808"
DATASET_ID = "ecommerce_analytics"

client = bigquery.Client(project=PROJECT_ID)
print("Connected:", PROJECT_ID)


Connected: ecommerce-analytics-505808


In [ ]:
dataset_ref = f"{PROJECT_ID}.{DATASET_ID}"
dataset = bigquery.Dataset(dataset_ref)
dataset.location = "US"
client.create_dataset(dataset, exists_ok=True)
print("Dataset ready:", dataset_ref)


## 1. Generate 2,000 customers

In [ ]:
rng = np.random.default_rng(42)

n_customers = 2000
customer_ids = [f"CUST{n:05d}" for n in range(1, n_customers + 1)]

first_names = [
    "Aarav","Aditi","Aditya","Ananya","Arjun","Diya","Ishaan","Isha",
    "Kabir","Kavya","Krishna","Meera","Neha","Nikhil","Priya","Rahul",
    "Riya","Rohan","Sanjay","Sneha","Tanvi","Varun","Vikram","Zoya"
]
last_names = [
    "Sharma","Patel","Reddy","Nair","Iyer","Kumar","Gupta","Singh",
    "Joshi","Rao","Verma","Shah","Das","Mishra","Kapoor","Mehta"
]

cities = [
    "Bengaluru","Mumbai","Delhi","Hyderabad","Chennai","Pune","Kolkata",
    "Ahmedabad","Jaipur","Kochi","Coimbatore","Mysuru","Noida","Gurugram"
]
states = {
    "Bengaluru":"Karnataka","Mumbai":"Maharashtra","Delhi":"Delhi",
    "Hyderabad":"Telangana","Chennai":"Tamil Nadu","Pune":"Maharashtra",
    "Kolkata":"West Bengal","Ahmedabad":"Gujarat","Jaipur":"Rajasthan",
    "Kochi":"Kerala","Coimbatore":"Tamil Nadu","Mysuru":"Karnataka",
    "Noida":"Uttar Pradesh","Gurugram":"Haryana"
}

city_values = rng.choice(cities, n_customers)
signup_dates = pd.to_datetime(
    rng.integers(
        pd.Timestamp("2024-01-01").value // 10**9,
        pd.Timestamp("2026-08-01").value // 10**9,
        n_customers
    ), unit="s"
)

customers = pd.DataFrame({
    "customer_id": customer_ids,
    "customer_name": [
        f"{rng.choice(first_names)} {rng.choice(last_names)}"
        for _ in range(n_customers)
    ],
    "age_group": rng.choice(
        ["18-24","25-34","35-44","45-54","55+"],
        n_customers, p=[0.16,0.34,0.25,0.17,0.08]
    ),
    "city": city_values,
    "state": [states[c] for c in city_values],
    "country": rng.choice(
        ["India","United States","United Kingdom","Singapore","UAE"],
        n_customers, p=[0.78,0.08,0.05,0.04,0.05]
    ),
    "signup_date": signup_dates,
    "customer_segment": rng.choice(
        ["New","Regular","Loyal","VIP"],
        n_customers, p=[0.28,0.45,0.20,0.07]
    )
})

print("Customers:", len(customers))
display(customers.head())


Customers: 2000


,customer_id,customer_name,age_group,city,state,country,signup_date,customer_segment
0,CUST00001,Ananya Sharma,25-34,Mumbai,Maharashtra,India,2026-04-08 01:51:00,VIP
1,CUST00002,Nikhil Iyer,25-34,Coimbatore,Tamil Nadu,India,2025-01-09 20:41:56,Regular
2,CUST00003,Krishna Reddy,55+,Kochi,Kerala,India,2025-03-08 03:23:11,Regular
3,CUST00004,Diya Singh,18-24,Kolkata,West Bengal,India,2026-01-22 09:36:19,Regular
4,CUST00005,Riya Gupta,35-44,Kolkata,West Bengal,India,2024-05-01 16:12:29,New


## 2. Generate 60 products

In [ ]:
categories = {
    "Electronics": ["Smartphones","Laptops","Headphones","Accessories"],
    "Home": ["Kitchen","Furniture","Decor","Appliances"],
    "Fashion": ["Men","Women","Footwear","Accessories"],
    "Beauty": ["Skincare","Haircare","Makeup","Personal Care"],
    "Sports": ["Fitness","Outdoor","Team Sports","Sportswear"]
}

brands = ["Nova","Apex","UrbanX","Zenith","Pulse","Vertex","Orion","VivoTech","HomePro","StyleHub"]

price_ranges = {
    "Electronics": (800, 85000),
    "Home": (300, 35000),
    "Fashion": (250, 12000),
    "Beauty": (150, 8000),
    "Sports": (300, 18000)
}

product_rows = []
product_num = 1

for category, subcats in categories.items():
    for subcat in subcats:
        for _ in range(3):
            low, high = price_ranges[category]
            selling_price = round(float(np.exp(rng.uniform(np.log(low), np.log(high)))), 2)
            unit_cost = round(selling_price * rng.uniform(0.50, 0.78), 2)

            product_rows.append({
                "product_id": f"PROD{product_num:04d}",
                "product_name": f"{rng.choice(brands)} {subcat} {product_num:02d}",
                "category": category,
                "sub_category": subcat,
                "brand": rng.choice(brands),
                "unit_cost": unit_cost,
                "selling_price": selling_price
            })
            product_num += 1

products = pd.DataFrame(product_rows)
print("Products:", len(products))
display(products.head())


Products: 60


,product_id,product_name,category,sub_category,brand,unit_cost,selling_price
0,PROD0001,Nova Smartphones 01,Electronics,Smartphones,Nova,15147.16,23572.74
1,PROD0002,StyleHub Smartphones 02,Electronics,Smartphones,Apex,1296.01,1701.09
2,PROD0003,Pulse Smartphones 03,Electronics,Smartphones,UrbanX,11180.07,15488.05
3,PROD0004,StyleHub Laptops 04,Electronics,Laptops,UrbanX,12905.56,23568.75
4,PROD0005,Vertex Laptops 05,Electronics,Laptops,UrbanX,2669.83,3427.24


## 3. Generate 20,000 orders

In [ ]:
n_orders = 20000

order_dates = pd.to_datetime(
    rng.integers(
        pd.Timestamp("2025-08-01").value // 10**9,
        pd.Timestamp("2026-09-01").value // 10**9,
        n_orders
    ), unit="s"
)

orders = pd.DataFrame({
    "order_id": [f"ORD{n:06d}" for n in range(1, n_orders + 1)],
    "order_date": order_dates,
    "customer_id": rng.choice(customers["customer_id"].values, n_orders),
    "product_id": rng.choice(products["product_id"].values, n_orders),
    "quantity": rng.choice([1,2,3,4,5], n_orders, p=[0.48,0.28,0.14,0.07,0.03]),
    "discount_pct": rng.choice([0,5,10,15,20,25], n_orders, p=[0.22,0.22,0.25,0.17,0.10,0.04]),
    "payment_method": rng.choice(
        ["UPI","Credit Card","Debit Card","Net Banking","COD","Wallet"],
        n_orders, p=[0.32,0.24,0.16,0.10,0.10,0.08]
    ),
    "sales_channel": rng.choice(
        ["Website","Mobile App","Marketplace"],
        n_orders, p=[0.48,0.37,0.15]
    ),
    "order_status": rng.choice(
        ["Completed","Returned","Cancelled"],
        n_orders, p=[0.88,0.07,0.05]
    )
})

orders = orders.merge(
    products[["product_id","unit_cost","selling_price"]],
    on="product_id", how="left"
)

orders["gross_amount"] = orders["quantity"] * orders["selling_price"]
orders["discount_amount"] = orders["gross_amount"] * orders["discount_pct"] / 100

orders["revenue"] = np.where(
    orders["order_status"].eq("Completed"),
    orders["gross_amount"] - orders["discount_amount"], 0
).round(2)

orders["cost"] = np.where(
    orders["order_status"].eq("Completed"),
    orders["quantity"] * orders["unit_cost"], 0
).round(2)

orders["profit"] = (orders["revenue"] - orders["cost"]).round(2)
orders = orders.sort_values("order_date").reset_index(drop=True)

print("Orders:", len(orders))
print("Revenue:", round(orders["revenue"].sum(), 2))
print("Profit:", round(orders["profit"].sum(), 2))
display(orders.head())


Orders: 20000
Revenue: 245970305.89
Profit: 68563180.55


,order_id,order_date,customer_id,product_id,quantity,discount_pct,payment_method,sales_channel,order_status,unit_cost,selling_price,gross_amount,discount_amount,revenue,cost,profit
0,ORD000055,2025-08-01 00:07:30,CUST00633,PROD0049,1,10,Net Banking,Website,Returned,752.59,1264.36,1264.36,126.436,0.00,0.00,0.00
1,ORD017507,2025-08-01 00:10:58,CUST00407,PROD0013,1,5,Net Banking,Mobile App,Completed,3843.09,6199.28,6199.28,309.964,5889.32,3843.09,2046.23
2,ORD007276,2025-08-01 00:24:28,CUST00414,PROD0006,3,0,UPI,Website,Completed,1109.83,1461.86,4385.58,0.000,4385.58,3329.49,1056.09
3,ORD011987,2025-08-01 00:28:52,CUST00025,PROD0005,2,15,Debit Card,Website,Completed,2669.83,3427.24,6854.48,1028.172,5826.31,5339.66,486.65
4,ORD017649,2025-08-01 01:02:02,CUST01500,PROD0042,3,0,Credit Card,Website,Completed,2830.75,4315.50,12946.50,0.000,12946.50,8492.25,4454.25


## 4. Replace the old BigQuery tables

In [ ]:
for table_name, df in {
    "customers": customers,
    "products": products,
    "orders": orders
}.items():
    table_id = f"{PROJECT_ID}.{DATASET_ID}.{table_name}"
    job_config = bigquery.LoadJobConfig(
        write_disposition="WRITE_TRUNCATE",
        autodetect=True
    )
    client.load_table_from_dataframe(df, table_id, job_config=job_config).result()
    print(f"Uploaded {len(df):,} rows -> {table_id}")


Uploaded 2,000 rows -> ecommerce-analytics-505808.ecommerce_analytics.customers
Uploaded 60 rows -> ecommerce-analytics-505808.ecommerce_analytics.products
Uploaded 20,000 rows -> ecommerce-analytics-505808.ecommerce_analytics.orders


## 5. Create analytics views for Data Studio

In [ ]:
sql = f'''
CREATE OR REPLACE VIEW `{PROJECT_ID}.{DATASET_ID}.vw_monthly_sales` AS
SELECT
  DATE_TRUNC(DATE(order_date), MONTH) AS month,
  COUNTIF(order_status = "Completed") AS completed_orders,
  SUM(IF(order_status = "Completed", quantity, 0)) AS units_sold,
  ROUND(SUM(revenue), 2) AS revenue,
  ROUND(SUM(profit), 2) AS profit
FROM `{PROJECT_ID}.{DATASET_ID}.orders`
GROUP BY month
ORDER BY month
'''
client.query(sql).result()
print("Created vw_monthly_sales")


Created vw_monthly_sales


In [ ]:
sql = f'''
CREATE OR REPLACE VIEW `{PROJECT_ID}.{DATASET_ID}.vw_product_performance` AS
SELECT
  p.product_id,
  p.product_name,
  p.category,
  p.sub_category,
  p.brand,
  SUM(IF(o.order_status = "Completed", o.quantity, 0)) AS units_sold,
  COUNTIF(o.order_status = "Completed") AS completed_orders,
  ROUND(SUM(o.revenue), 2) AS revenue,
  ROUND(SUM(o.profit), 2) AS profit,
  ROUND(SAFE_DIVIDE(SUM(o.profit), SUM(o.revenue)) * 100, 2) AS profit_margin_pct
FROM `{PROJECT_ID}.{DATASET_ID}.products` p
LEFT JOIN `{PROJECT_ID}.{DATASET_ID}.orders` o
ON p.product_id = o.product_id
GROUP BY p.product_id, p.product_name, p.category, p.sub_category, p.brand
ORDER BY revenue DESC
'''
client.query(sql).result()
print("Created vw_product_performance")


Created vw_product_performance


In [ ]:
sql = f'''
CREATE OR REPLACE VIEW `{PROJECT_ID}.{DATASET_ID}.vw_customer_summary` AS
SELECT
  c.customer_id,
  c.customer_name,
  c.age_group,
  c.city,
  c.state,
  c.country,
  c.customer_segment,
  COUNTIF(o.order_status = "Completed") AS completed_orders,
  ROUND(SUM(o.revenue), 2) AS revenue,
  ROUND(SUM(o.profit), 2) AS profit,
  ROUND(SAFE_DIVIDE(SUM(o.revenue), COUNTIF(o.order_status = "Completed")), 2) AS average_order_value
FROM `{PROJECT_ID}.{DATASET_ID}.customers` c
LEFT JOIN `{PROJECT_ID}.{DATASET_ID}.orders` o
ON c.customer_id = o.customer_id
GROUP BY c.customer_id, c.customer_name, c.age_group, c.city, c.state, c.country, c.customer_segment
ORDER BY revenue DESC
'''
client.query(sql).result()
print("Created vw_customer_summary")


Created vw_customer_summary


In [ ]:
sql = f'''
CREATE OR REPLACE VIEW `{PROJECT_ID}.{DATASET_ID}.vw_category_performance` AS
SELECT
  p.category,
  p.sub_category,
  SUM(IF(o.order_status = "Completed", o.quantity, 0)) AS units_sold,
  COUNTIF(o.order_status = "Completed") AS completed_orders,
  ROUND(SUM(o.revenue), 2) AS revenue,
  ROUND(SUM(o.profit), 2) AS profit,
  ROUND(SAFE_DIVIDE(SUM(o.profit), SUM(o.revenue)) * 100, 2) AS profit_margin_pct
FROM `{PROJECT_ID}.{DATASET_ID}.products` p
LEFT JOIN `{PROJECT_ID}.{DATASET_ID}.orders` o
ON p.product_id = o.product_id
GROUP BY p.category, p.sub_category
ORDER BY revenue DESC
'''
client.query(sql).result()
print("Created vw_category_performance")


Created vw_category_performance


## 6. Verify everything

In [ ]:
kpis = client.query(f'''
SELECT
  COUNT(*) AS total_orders,
  COUNTIF(order_status = "Completed") AS completed_orders,
  COUNTIF(order_status = "Returned") AS returned_orders,
  COUNTIF(order_status = "Cancelled") AS cancelled_orders,
  ROUND(SUM(revenue), 2) AS total_revenue,
  ROUND(SUM(profit), 2) AS total_profit,
  ROUND(SAFE_DIVIDE(SUM(profit), SUM(revenue)) * 100, 2) AS profit_margin_pct,
  ROUND(SAFE_DIVIDE(SUM(revenue), COUNTIF(order_status = "Completed")), 2) AS average_order_value
FROM `{PROJECT_ID}.{DATASET_ID}.orders`
''').to_dataframe()

display(kpis)

print("Monthly sales:")
display(client.query(f'''
SELECT * FROM `{PROJECT_ID}.{DATASET_ID}.vw_monthly_sales`
ORDER BY month
''').to_dataframe())

print("Top 10 products:")
display(client.query(f'''
SELECT * FROM `{PROJECT_ID}.{DATASET_ID}.vw_product_performance`
ORDER BY revenue DESC LIMIT 10
''').to_dataframe())


,total_orders,completed_orders,returned_orders,cancelled_orders,total_revenue,total_profit,profit_margin_pct,average_order_value
0,20000,17645,1436,919,2.459703e+08,68563180.55,27.87,13939.94


Monthly sales:


,month,completed_orders,units_sold,revenue,profit
0,2025-08-01,1386,2582,19626779.83,5453574.45
1,2025-09-01,1325,2509,18863072.33,5315663.97
2,2025-10-01,1383,2583,19749246.10,5444413.06
3,2025-11-01,1313,2478,16126614.69,4595475.76
4,2025-12-01,1348,2516,19253517.26,5383632.83
5,2026-01-01,1336,2570,20881318.81,5834781.36
6,2026-02-01,1238,2309,16968593.41,4648793.97
7,2026-03-01,1362,2624,18018812.39,5122203.03
8,2026-04-01,1373,2584,18023167.45,5098074.62
9,2026-05-01,1400,2633,19819859.08,5475029.74


Top 10 products:


,product_id,product_name,category,sub_category,brand,units_sold,completed_orders,revenue,profit,profit_margin_pct
0,PROD0009,Zenith Headphones 09,Electronics,Headphones,HomePro,520,282,39990510.05,6317665.65,15.80
1,PROD0007,Vertex Headphones 07,Electronics,Headphones,Vertex,639,322,28182816.23,7088429.39,25.15
2,PROD0020,UrbanX Decor 20,Home,Decor,Apex,578,302,15205670.04,5806239.82,38.18
3,PROD0008,StyleHub Headphones 08,Electronics,Headphones,Apex,553,303,13412376.06,3051351.47,22.75
4,PROD0001,Nova Smartphones 01,Electronics,Smartphones,Nova,611,302,13075799.00,3820884.24,29.22
5,PROD0004,StyleHub Laptops 04,Electronics,Laptops,UrbanX,447,241,9603087.22,3834301.90,39.93
6,PROD0003,Pulse Smartphones 03,Electronics,Smartphones,UrbanX,561,291,7880319.65,1608300.38,20.41
7,PROD0012,Orion Accessories 12,Electronics,Accessories,Pulse,499,266,7832416.22,2715595.37,34.67
8,PROD0053,HomePro Outdoor 53,Sports,Outdoor,HomePro,609,312,6803334.95,1342310.15,19.73
9,PROD0010,Zenith Accessories 10,Electronics,Accessories,Vertex,500,282,6683637.70,1968312.70,29.45


## Data Studio dashboard after refresh

Use these views:
- `vw_monthly_sales` → revenue/profit trend
- `vw_product_performance` → top products and profitability
- `vw_customer_summary` → customer segments and AOV
- `vw_category_performance` → category revenue/profit

Recommended KPI cards:
**Total Revenue | Total Orders | Total Profit | Profit Margin | Average Order Value | Units Sold | Return Rate**
